# tp-05k 専用化の効果を、アンサンブル効果と切り分ける（対照実験）

tp-05j では、区分ごとに別モデルを作る（separate）と悪化し、全学習行を使って区分外の重みを下げる（weighted）は中立、weighted と全体モデルの 50:50 平均だけが小さく改善した。
ただし 50:50 平均は「別々に学習した 2 モデルの平均」なので、**専用化の効果ではなく単なるアンサンブル（分散低減）の効果かもしれない**。基準の全体モデルは 1 seed だった。

1. **対照**: 全体モデル seed 0 と seed 1 の 50:50 平均（専用化なしの同じ効果）。専用 weighted（seed 0）+ 全体（seed 0）の平均が、この対照を上回るか。
2. **アンサンブル v2 への追加**: 専用 weighted の OOF（クラス・コース）を候補に加えて LOYO で合成し直し、v2（勝率 0.21394 / 連対 0.33615 / 複勝 0.42227）を有意に上回るか。

採用基準 = 差の 95% 信頼区間の上限 < 0 かつ連対・複勝が悪化しない（0.0002 以内）。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import segment_models as sm, tp05_data, win_models as wm, zoo, zoo_runner, ensemble as en
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_k"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP05K_THREADS", detect_compute(reserve_cores=6).threads))
mlflow.set_experiment("keiba-tp-segment")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
SEGS = {"grade": sm.grade_bucket(df["race_class"], df["surface"]).to_numpy(), "course": np.asarray(sm.course_key(df["surface"], df["distance"]))}
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
res0, G0 = zoo_runner.run_oof("lgb_global_s0", "lgb_binary", X, meta, folds, [0], THREADS, params=dict(LP))
res1, G1 = zoo_runner.run_oof("lgb_global_s1", "lgb_binary", X, meta, folds, [1], THREADS, params=dict(LP))
print(f"全体 seed0 win {res0['logloss_win'].mean():.5f} / seed1 {res1['logloss_win'].mean():.5f}")


def mix(a, b):
    m = a.merge(b, on=key, suffixes=("_a", "_b"))
    out = m[key + ["year_a"]].rename(columns={"year_a": "year"}).copy()
    for t, tot in (("win", 1.0), ("top2", 2.0), ("top3", 3.0)):
        out[f"p_{t}"] = wm.normalize_to_total(m["race_id"].to_numpy(), 0.5 * m[f"p_{t}_a"].to_numpy() + 0.5 * m[f"p_{t}_b"].to_numpy(), tot)
    return out


def compare(new, ref, name):
    m = ref.merge(new, on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": name}
    for t, k in (("win", 1), ("top2", 2), ("top3", 3)):
        yy = (fin <= k).astype(float)
        bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t}_b"].to_numpy(), yy), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{t}_v"].to_numpy(), yy), n_boot=1000)
        out[f"{t}_base"], out[f"{t}_new"], out[f"Δ{t}"], out[f"{t}_hi"] = wm.binary_logloss(m[f"p_{t}_b"].to_numpy(), yy), wm.binary_logloss(m[f"p_{t}_v"].to_numpy(), yy), bb["diff"], bb["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 2e-4 and out["Δtop3"] <= 2e-4)
    return out


CTRL = mix(G0, G1)
ROWS = [compare(CTRL, G0, "対照: 全体 seed0 + 全体 seed1 の平均 vs 全体 seed0")]
display(pd.DataFrame(ROWS).set_index("variant").round(5))

In [ ]:
SPEC = {}
for fam in ("course", "grade"):
    t0 = time.time()
    oof, _ = sm.oof_segmented(X, meta, folds, SEGS[fam], LP["lgb"], LP["n_rounds"], THREADS, mode="weighted", weight_other=0.25, base_oof=G0)
    oof.to_parquet(OOF_DIR / f"lgb_{fam}_weighted.parquet", index=False)
    SPEC[fam] = oof
    mx = mix(G0, oof)
    ROWS.append(compare(mx, G0, f"{fam} weighted + 全体 seed0 の平均 vs 全体 seed0"))
    ROWS.append(compare(mx, CTRL, f"{fam} weighted + 全体 seed0 の平均 vs 対照（全体 seed0+seed1）"))
    print(fam, f"{time.time() - t0:.0f}s", flush=True)
    gc.collect()
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_base", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5)); R.to_csv(OUT / "control.csv")

## アンサンブル v2 に専用 weighted を加える（LOYO）
候補 = v2 の 7 モデル + 専用 weighted（クラス・コース）。v2 の OOF との paired bootstrap。

In [ ]:
v2 = json.loads((WORK / "ensemble_v2/ensemble_spec.json").read_text(encoding="utf-8"))
names = v2["models"] + ["lgb_course_weighted", "lgb_grade_weighted"]
base = meta[key + ["finish_position", "year"]].merge(pd.read_parquet(OOF_DIR / f"{names[0]}.parquet")[key], on=key, how="inner")
for n in names:
    o = pd.read_parquet(OOF_DIR / f"{n}.parquet")[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")})
    base = base.merge(o, on=key, how="inner")
race, years, fin = base["race_id"].to_numpy(), base["year"].to_numpy(), base["finish_position"].to_numpy()
TOT = {"win": 1.0, "top2": 2.0, "top3": 3.0}
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
P = {t: np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1) for t in TOT}
v2oof = base[key].merge(pd.read_parquet(OOF_DIR / "ensemble_v2.parquet")[key + ["p_win", "p_top2", "p_top3"]], on=key, how="left")
ORDER = ["mean", "logit_mean", "greedy", "weighted_logit", "stack_lr"]
rows, final_P = [], {}
for t in TOT:
    sub = {}
    outs = {}
    for m_ in en.METHODS:
        v, out = en.loyo_eval(m_, P[t], years, race, Y[t], TOT[t])
        sub[m_], outs[m_] = v, out
    ch = next(m_ for m_ in ORDER if sub[m_] <= min(sub[x] for x in ORDER) + 5e-5)
    final_P[t] = outs[ch]
    b = wm.paired_bootstrap_diff(wm.per_race_loss(race, v2oof[f"p_{t}"].to_numpy(), Y[t]), wm.per_race_loss(race, final_P[t], Y[t]), n_boot=1000)
    rows.append({"target": t, "v2": en.ll(v2oof[f"p_{t}"].to_numpy(), Y[t]), "v2+専用weighted": en.ll(final_P[t], Y[t]), "合成案": ch, "Δ": b["diff"], "hi": b["hi"], "採用": b["hi"] < 0})
ENS = pd.DataFrame(rows).set_index("target")
display(ENS.round(5)); ENS.to_csv(OUT / "ensemble_with_segment.csv")
ADOPT = bool(ENS.loc["win", "採用"] and ENS.loc["top2", "Δ"] <= 2e-4 and ENS.loc["top3", "Δ"] <= 2e-4)
print("v2 に専用 weighted を加える案を採用:", ADOPT)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "control": R.round(6).reset_index().to_dict("records"), "ensemble": ENS.round(6).reset_index().to_dict("records"), "adopt": ADOPT}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-05k-segment-control", tags={"notebook": "tp-05k", "git_commit": GIT_COMMIT}):
    mlflow.log_metrics({f"ens_delta_{t}": float(ENS.loc[t, "Δ"]) for t in TOT})
    for f_ in ("control.csv", "ensemble_with_segment.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_k")